# Membrane Protein Data - Data Cleaning

## Objectives

* Load and inspect the membrane protein dataset obtained from the OPM database.
* Assess the structure and quality of the dataset.
* Identify missing values, duplicate records, and inappropriate data types.
* Clean and transform the data where necessary, documenting the reason for each decision.
* Prepare a cleaned dataset for exploratory data analysis.

## Inputs

'data/proteins-2026-08-10.csv` - raw membrane protein structural data downloaded from the OPM database.

## Outputs
'data/cleaned/proteins_cleaned.csv' - cleaned membrane protein dataset prepared for exploratory data analysis. 

Additional Comments

The raw dataset will be preserved unchanged.
Cleaning decisions will be based on the relevance and quality of individual variables rather than automatically removing all records containing missing values.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os

project_dir = r"C:\Users\trasn\OneDrive\Dokumente\programing\37 week-app\membrane-protein-data-analysis"

os.chdir(project_dir)

print(os.getcwd())

C:\Users\trasn\OneDrive\Dokumente\programing\37 week-app\membrane-protein-data-analysis


# Section 1
Data Collection and Initial Inspection


The raw OPM membrane protein dataset will first be loaded and inspected to understand its dimensions, variables, data types, and overall structure before any cleaning decisions are made.

## Load the Dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/proteins-2026-08-10.csv")
df.head()


,id,ordering,family_name_cache,species_name_cache,membrane_name_cache,name,description,comments,pdbid,resolution,...,superfamily_id,classtype_id,type_id,secondary_representations_count,structure_subunits_count,citations_count,created_at,updated_at,uniprotcode,interpro
0,1,6024.0,OmpA family,Escherichia coli,Gram-neg. outer,"Outer membrane protein A (OMPA), disordered loops",NaN,OmpA is required for the action of colicins K ...,"=""1qjp""",1.65,...,26,2,1,3,1,2,2018-08-13 03:49:46 UTC,2023-02-22 20:43:56 UTC,OMPA_ECOLI,NaN
1,2,6027.0,Enterobacterial Ail/Lom protein,Escherichia coli,Gram-neg. outer,Outer membrane protein X (OMPX),NaN,OmpX from Escherichia coli promotes adhesion t...,"=""1qj8""",1.9,...,26,2,1,7,1,1,2018-08-13 03:49:46 UTC,2023-02-22 20:43:57 UTC,OMPX_ECOLI,NaN
2,3,6032.0,Opacity porins,Neisseria meningitidis,Gram-neg. outer,Outer membrane protein NspA,NaN,Pathogenic Neisseria spp. possess a repertoire...,"=""1p4t""",2.55,...,235,2,1,0,1,0,2018-08-13 03:49:46 UTC,2023-02-22 20:43:57 UTC,Q9RP17_NEIME,NaN
3,4,5624.0,Influenza virus matrix protein 2,Influenza A virus,Viral,"M2 proton channel of Influenza A, closed state...",NaN,NaN,"=""3lbw""",1.65,...,185,11,1,3,4,0,2018-08-13 03:49:46 UTC,2023-02-22 20:43:54 UTC,M2_I97A1,NaN
4,5,6047.0,"OM protease omptin, OMPT",Yersinia pestis,Gram-neg. outer,Plasminogen activator PLA (coagulase/fibrinoly...,NaN,NaN,"=""2x55""",1.85,...,27,2,1,2,1,0,2018-08-13 03:49:46 UTC,2023-02-22 20:43:56 UTC,COLY_YERPE,NaN


---

## Dataset Dimensions

Before cleaning the data, the size of the dataset is checked to understand how many observations and variables are available for analysis.

In [3]:
df.shape

(8915, 33)

The raw dataset contains 8,915 observations and 33 variables. This provides a sufficiently large dataset for exploring patterns in the structural and biological characteristics of membrane proteins.

## Column Names

The column names are inspected to understand the variables available in the dataset and to identify which may be relevant to the membrane protein analysis.

In [4]:
df.columns

Index(['id', 'ordering', 'family_name_cache', 'species_name_cache',
       'membrane_name_cache', 'name', 'description', 'comments', 'pdbid',
       'resolution', 'topology_subunit', 'topology_show_in', 'thickness',
       'thicknesserror', 'subunit_segments', 'tilt', 'tilterror', 'gibbs',
       'tau', 'verification', 'membrane_id', 'species_id', 'family_id',
       'superfamily_id', 'classtype_id', 'type_id',
       'secondary_representations_count', 'structure_subunits_count',
       'citations_count', 'created_at', 'updated_at', 'uniprotcode',
       'interpro'],
      dtype='object')

### Initial observation

The dataset contains 33 variables representing a mixture of protein identifiers, biological classifications, structural measurements, database relationships, and metadata.

Variables such as `resolution`, `thickness`, `subunit_segments`, `tilt`, and `gibbs` appear particularly relevant for structural analysis, while `family_name_cache`, `species_name_cache`, and `membrane_name_cache` may allow comparisons between biological groups.

Other columns, such as database IDs and timestamps, may be useful for identification or linking records but are not necessarily direct biological measurements.

## Data Types and Dataset Structure

The structure of the dataset is examined to identify the data type of each variable and the number of non-null values available. This will help identify potential data quality issues that require further investigation.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8915 entries, 0 to 8914
Data columns (total 33 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id                               8915 non-null   int64  
 1   ordering                         8915 non-null   float64
 2   family_name_cache                8915 non-null   object 
 3   species_name_cache               8915 non-null   object 
 4   membrane_name_cache              8915 non-null   object 
 5   name                             8915 non-null   object 
 6   description                      0 non-null      float64
 7   comments                         1202 non-null   object 
 8   pdbid                            8915 non-null   object 
 9   resolution                       8879 non-null   object 
 10  topology_subunit                 6653 non-null   object 
 11  topology_show_in                 8915 non-null   bool   
 12  thickness           

### Initial observations on dataset structure

The dataset contains a mixture of numerical, categorical, Boolean, and text variables. Several potential data quality issues are visible from the initial inspection.

- `description` and `interpro` contain no non-null values.
- `tau` and `verification` contain data for only a small proportion of the 8,915 records.
- `comments` and `topology_subunit` also contain substantial missing data.
- `resolution` contains 8,879 non-null values and is stored as an `object` rather than a numerical data type, which requires further investigation.
- `uniprotcode` is available for most, but not all, records.
- `created_at` and `updated_at` are stored as `object` values rather than datetime values.

These observations will be investigated further before making any cleaning decisions.

## Numerical Summary

Summary statistics are examined for the numerical variables to understand their ranges, central tendencies, and potential unusual values before data cleaning.

In [6]:
df.describe()

,id,ordering,description,thickness,thicknesserror,subunit_segments,tilt,tilterror,gibbs,tau,membrane_id,species_id,family_id,superfamily_id,classtype_id,type_id,secondary_representations_count,structure_subunits_count,citations_count,interpro
count,8915.000000,8915.000000,0.0,8915.000000,8880.000000,8915.000000,8915.000000,8886.000000,8915.000000,150.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,8915.000000,0.0
mean,4976.208749,4458.000000,NaN,22.218912,1.242827,14.132698,23.628828,3.444182,-89.141963,196.933333,7.729557,168.035895,402.484577,115.438811,2.869546,1.360067,0.892653,3.019966,0.039596,NaN
std,3127.510996,2573.683158,NaN,12.084948,2.387358,18.642101,30.735730,12.673102,88.521995,161.436283,6.728683,263.549342,369.247487,144.125066,3.019443,0.635339,4.620441,5.572643,0.347920,NaN
min,1.000000,1.000000,NaN,0.000000,-3.000000,0.000000,0.000000,-4.000000,-691.000000,60.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,NaN
25%,2291.500000,2229.500000,NaN,6.500000,0.600000,0.000000,1.000000,0.000000,-126.900000,90.000000,4.000000,14.000000,60.000000,8.000000,1.000000,1.000000,0.000000,0.000000,0.000000,NaN
50%,4576.000000,4458.000000,NaN,29.600000,1.000000,10.000000,7.000000,1.000000,-71.500000,110.000000,4.000000,36.000000,278.000000,50.000000,1.000000,1.000000,0.000000,1.000000,0.000000,NaN
75%,8103.500000,6686.500000,NaN,31.000000,1.500000,21.000000,44.000000,4.000000,-9.600000,250.000000,9.000000,213.000000,724.000000,179.000000,4.000000,2.000000,0.000000,4.000000,0.000000,NaN
max,10363.000000,8915.000000,NaN,41.800000,180.000000,389.000000,91.000000,1039.000000,165.500000,600.000000,24.000000,1150.000000,1240.000000,607.000000,11.000000,3.000000,132.000000,70.000000,11.000000,NaN


### Initial observations on numerical variables

The numerical summary shows substantial variation across several structural variables. Membrane thickness has a median of 29.6, while the number of subunit segments ranges from 0 to 389. Gibbs energy also shows a wide range of values.

Some variables contain potentially unusual values. For example, `thicknesserror` and `tilterror` have maximum values substantially higher than their upper quartiles. These values should be investigated before deciding whether they represent valid observations or data quality issues.

The `description` and `interpro` columns contain no numerical observations, while `tau` contains only 150 observations. Their usefulness will therefore require further assessment during data cleaning.

The `resolution` variable is absent from the numerical summary because it is currently stored as an `object` data type. As structural resolution is expected to represent a numerical measurement, the contents of this column will be investigated during data cleaning.

# Section 2

# Section 2: Data Cleaning
The raw dataset will be assessed for missing values, duplicate records, inappropriate data types, and variables with limited analytical value. Cleaning decisions will be made based on the characteristics of the data and the objectives of the analysis.

## Missing Values

Missing values are examined to determine the completeness of each variable and to identify columns that may require removal, transformation, or further investigation.

In [7]:
df.isnull().sum()

id                                    0
ordering                              0
family_name_cache                     0
species_name_cache                    0
membrane_name_cache                   0
name                                  0
description                        8915
comments                           7713
pdbid                                 0
resolution                           36
topology_subunit                   2262
topology_show_in                      0
thickness                             0
thicknesserror                       35
subunit_segments                      0
tilt                                  0
tilterror                            29
gibbs                                 0
tau                                8765
verification                       8693
membrane_id                           0
species_id                            0
family_id                             0
superfamily_id                        0
classtype_id                          0


---

In [8]:
missing_values = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values

,Missing_Count,Missing_Percentage
id,0,0.00
ordering,0,0.00
family_name_cache,0,0.00
species_name_cache,0,0.00
membrane_name_cache,0,0.00
name,0,0.00
description,8915,100.00
comments,7713,86.52
pdbid,0,0.00
resolution,36,0.40


### Missing Values Observations

The analysis shows that missing data are not distributed equally across the dataset.

- `description` and `interpro` are completely empty, with 100% missing values.
- `tau` and `verification` are almost entirely missing, with 98.32% and 97.51% missing values respectively.
- `comments` contains 86.52% missing values.
- `topology_subunit` contains 25.37% missing values.
- `uniprotcode` contains 3.75% missing values.
- The structural variables `resolution`, `thicknesserror`, and `tilterror` contain less than 0.5% missing data.

Different levels of missingness require different treatment, and the biological or analytical relevance of each variable should be considered before removing data.

## Removing Empty Columns

The `description` and `interpro` columns contain no observations and therefore provide no information for the analysis. They will be removed from the working dataset.

In [9]:
df_clean = df.copy()

df_clean = df_clean.drop(columns=["description", "interpro"])

df_clean.shape

(8915, 31)

The two completely empty columns were removed, reducing the dataset from 33 to 31 variables while retaining all 8,915 protein records.

## Investigation of Sparse Columns

Several columns contain a high proportion of missing values. Before deciding whether to remove them, the available values are inspected to determine whether they provide useful information for the objectives of this analysis.

In [10]:
df_clean[["tau", "verification", "comments"]].count()





tau              150
verification     222
comments        1202
dtype: int64

In [11]:
df_clean["tau"].dropna().head(10)

33      80.0
109    130.0
189    250.0
190    150.0
191     80.0
192    120.0
193     80.0
194    110.0
195    270.0
196    100.0
Name: tau, dtype: float64

In [12]:
df_clean["verification"].dropna().head(10)



0     Four interfacial Trp residues of OmpA are loca...
1     Locations of the hydrophobic boundaries are co...
8     Membrane boundary planes of monomeric form (1q...
11    Results are consistent with experimental hydro...
19    Average tilt of TM beta-strands (38Â°) is slig...
20    Locations of hydrophobic boundaries are consis...
22    Locations of hydrophobic boundaries are consis...
27    Several Trp residues of alpha-hemolysin are si...
28    The calculated intrinsic hydrophobic thickness...
34    Calculated membrane core boundaries of MscL se...
Name: verification, dtype: object

In [13]:
df_clean["comments"].dropna().head(10)

0     OmpA is required for the action of colicins K ...
1     OmpX from Escherichia coli promotes adhesion t...
2     Pathogenic Neisseria spp. possess a repertoire...
6     Neisseria species specific OpcA proteins play ...
19    This receptor binds the ferrichrome-iron ligan...
22    Involved in the active translocation of vitami...
33    This is an open or another expanded state of t...
41    Part of the ABC transporter complex btuCDF inv...
47    Loop to helix transition in 50-residue N-termi...
55    The calculations are conducted for one half of...
Name: comments, dtype: object

### Decision on Sparse Variables

Inspection of the available values showed that the sparse columns contain meaningful information, but they are not suitable for the planned structured analysis.

- `tau` contains numerical information for only 150 of 8,915 records (1.68%), providing insufficient coverage for dataset-wide comparisons.
- `verification` contains scientifically relevant descriptive information, but is available for only 222 records (2.49%) and consists of unstructured text.
- `comments` contains useful biological and structural annotations, but is available for only 1,202 records (13.48%) and is also unstructured text.

As the project focuses on structured quantitative and categorical analysis rather than text analysis, these three variables will not be included in the analytical dataset.

In [14]:
df_clean = df_clean.drop(
    columns=["tau", "verification", "comments"]
)

df_clean.shape

(8915, 28)

## Remaining Missing Values

After removing the empty and highly sparse variables that are outside the scope of the analysis, the remaining dataset is reassessed to identify missing values that still require consideration.

In [15]:
remaining_missing = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": (df_clean.isnull().sum() / len(df_clean) * 100).round(2)
})

remaining_missing[remaining_missing["Missing_Count"] > 0]

,Missing_Count,Missing_Percentage
resolution,36,0.40
topology_subunit,2262,25.37
thicknesserror,35,0.39
tilterror,29,0.33
uniprotcode,334,3.75


### Observation

Five variables still contain missing values. The amount of missing data varies considerably, from less than 0.5% for `resolution`, `thicknesserror`, and `tilterror`, to 25.37% for `topology_subunit`.

These variables will be investigated individually because their analytical importance and the reasons for missing values may differ.

## Investigation of the Resolution Variable

Structural resolution is expected to be numerical, but the initial dataset inspection showed that `resolution` is stored as an object data type. The values are therefore inspected before any conversion or treatment of missing data is performed.

In [16]:
df_clean["resolution"].head(20)

0     1.65
1      1.9
2     2.55
3     1.65
4     1.85
5      1.5
6     2.03
7      2.6
8      2.1
9     3.01
10     1.9
11     2.0
12     3.2
13    1.45
14     1.8
15     1.9
16     2.4
17     2.4
18     2.4
19     2.5
Name: resolution, dtype: object

In [17]:
df_clean["resolution"].unique()

array(['1.65', '1.9', '2.55', '1.85', '1.5', '2.03', '2.6', '2.1', '3.01',
       '2.0', '3.2', '1.45', '1.8', '2.4', '2.5', '2.73', '2.56', '1.89',
       '2.51', '2.8', '3.0', '3.7', '3.5', '1.4', '3.3', '2.72', '2.2',
       '2.3', '2.45', '1.6', '2.7', '6.2 EM', '3.65', '2.9', 'NMR',
       '1.47', '1.93', '3.4', '3.1', '2.35', '1.9 EM', '2.24', '4.00',
       '2.65', '3.02', '1.75', '2.08', '1.83', '1.2', '2.19', '1.41',
       '1.7', '1.04', '1.15', '3.45', '3.54 EM', '1.72', '3.9', '1.68',
       '3.3 FD', '5.0 FD', '3.1 FD', '2.4 FD', '1.74', '1.78', '2.05',
       '3.2 EM', '1.3', '1.25', '1.46', '2.59', '1.42', '9.6 EM', '2.76',
       '2.13', '1.61', '2.18', '0.85', '1.79', '1.1', '0.97', '1.97',
       '1.36', '0.99', '2.15', '1.71', '1.0', '2.27', nan, '3.8', '3.82',
       '0.98', '3.06', '2.57', '1.38', '2.31', '2.39', '2.78', '1.16',
       '2.46', '1.55', '3.6', '1.82', '2.91', '3.52', '3.88', '1.57',
       '2.02', '2.54', '2.01', '2.95', '2.06', '0.54', '0.9', '0.95'

### Resolution Data Format

Inspection of the unique values shows that `resolution` contains a mixture of numerical values and text annotations such as `EM`, `EC`, `FD`, `NMR`, and `ND`. Some records also contain additional whitespace.

This mixed formatting explains why Pandas imported the column as an `object`. The different formats will be investigated before converting the resolution values to a numerical data type.

In [18]:
resolution_numeric = pd.to_numeric(
    df_clean["resolution"],
    errors="coerce"
)

df_clean.loc[
    resolution_numeric.isna() & df_clean["resolution"].notna(),
    ["pdbid", "name", "resolution"]
].head(20)

,pdbid,name,resolution
56,"=""4aq9""","Nicotinic acetylcholine receptor, partially op...",6.2 EM
66,"=""1a91""","F0 ATP synthase, subunit c",NMR
67,"=""1c17""",F0 ATP synthase,NMR
78,"=""2jo1""","Na,K-ATPase regulatory protein FXYD1 (phosphol...",NMR
87,"=""2b6o""",Aquaporin-0,1.9 EM
91,"=""1afo""",Glycophorin A,NMR
98,"=""1grm""","Gramicidin A, head-to-head dimer, right-handed",NMR
100,"=""1bh4""",Circulin A,NMR
101,"=""1myn""",Drosomycin,NMR
105,"=""1z65""",N-terminal helix of prion-like protein doppel,NMR


### Resolution Format Observation

The inspection confirms that the `resolution` column combines numerical resolution values with additional text annotations. For example, values such as `6.2 EM` contain both a numerical resolution and an experimental annotation, while entries such as `NMR` contain no numerical resolution value.

Therefore, directly converting the existing column to a numerical data type would result in the loss of valid numerical information. The numerical component should first be extracted before conversion.

In [19]:
df_clean["resolution_numeric"] = (
    df_clean["resolution"]
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

df_clean[
    ["pdbid", "resolution", "resolution_numeric"]
].head(100)

,pdbid,resolution,resolution_numeric
0,"=""1qjp""",1.65,1.65
1,"=""1qj8""",1.9,1.90
2,"=""1p4t""",2.55,2.55
3,"=""3lbw""",1.65,1.65
4,"=""2x55""",1.85,1.85
...,...,...,...
95,"=""1t5s""",2.6,2.60
96,"=""2zbd""",2.4,2.40
97,"=""1p49""",2.6,2.60
98,"=""1grm""",NMR,NaN


### Resolution Conversion Result

The numerical component of the `resolution` column was successfully extracted into a new variable, `resolution_numeric`.

Entries containing numerical values were converted correctly, including values that also contained text annotations. Entries without a numerical resolution, such as `NMR`, resulted in missing values in the new numerical column.

The original `resolution` column was retained to preserve the source information.

In [20]:
df_clean["resolution_numeric"].isnull().sum()

1339

In [21]:
(
    df_clean["resolution_numeric"].isnull().sum()
    / len(df_clean)
    * 100
).round(2)

15.02

### Missing Numerical Resolution

After extracting the numerical component, 1,339 records (15.02%) do not contain a usable numerical resolution value.

This is substantially higher than the 36 values originally identified as missing because some populated entries contain only text annotations, such as `NMR`, rather than a numerical resolution.

These records will be retained in the dataset because they may still contain useful information for other analyses. They can be excluded only when an analysis specifically requires numerical resolution.

In [22]:
df_clean["resolution_numeric"].describe()

count    7576.000000
mean        3.064163
std         1.262766
min         0.540000
25%         2.440000
50%         3.000000
75%         3.500000
max        37.000000
Name: resolution_numeric, dtype: float64

### Resolution Distribution

The extracted numerical resolution values are available for 7,576 protein records. The median resolution is 3.0, while 50% of the observations lie between 2.44 and 3.50.

The maximum value of 37.0 is substantially higher than the upper quartile and may represent an extreme but valid observation. Extreme resolution values will therefore be inspected before deciding whether any should be removed.

In [23]:
df_clean[
    ["pdbid", "name", "resolution", "resolution_numeric"]
].sort_values(
    by="resolution_numeric",
    ascending=False
).head(20)

,pdbid,name,resolution,resolution_numeric
1945,"=""4b2q""","F1F0 ATP synthase, structure 2",37.0 EM,37.0
3047,"=""5lcb""","Bacteriochlorophyll c-binding protein, complex...",26.5 EM,26.5
2174,"=""3j41""","Aquaporin-0, complex with calmodulin",25.0 EM,25.0
3138,"=""1k4r""","Envelope glycoprotein, chimeric",24.0 EM,24.0
1305,"=""2ybb""",Respiratory complex I,19.0 EM,19.0
3474,"=""5xti""",Mitochondrial respiratory supercomplex I2-III2...,17.4 EM,17.4
6898,"=""7o01""",Photosystem I,17.1 EM,17.1
6158,"=""4ckh""","ACAP1, tetramer",17.0 EM,17.0
2197,"=""3j2s""","Coagulation factor VIII, light chain, structure 1",15.0 EM,15.0
3230,"=""5mg3""",Holo-translocon,14.0 EM,14.0


### Resolution Outlier Assessment

Inspection of the highest numerical resolution values showed that the extreme observations correspond to entries labelled `EM` in the original dataset. For example, the maximum value of 37.0 was derived from an original entry of `37.0 EM`.

These observations therefore appear to represent genuine values in the source dataset rather than errors introduced during numerical extraction. Although they are statistical outliers, they will be retained because there is no evidence that they are invalid.

If resolution is used in later visualisations, the influence of these extreme values will be considered when interpreting the results.

In [24]:
df_clean["topology_subunit"].value_counts(dropna=False)

topology_subunit
A       5085
NaN     2262
R        414
B        295
C        198
D         90
E         64
L         52
a         41
K         37
F         34
G         33
X         31
H         30
1         28
S         27
I         27
P         18
J         17
0         17
Y         15
Q         15
O          9
A in       8
U          7
b          7
N          6
Z          6
8          5
M          5
T          4
2          4
c          4
W          2
x          2
y          2
o          2
r          1
4          1
6          1
3          1
5          1
p          1
n          1
h          1
7          1
g          1
l          1
V          1
Name: count, dtype: int64

### Topology Subunit Inspection

The `topology_subunit` variable contains mainly chain or subunit identifiers. The most common value is `A`, while other entries include additional letters, lowercase letters, numbers, and a small number of less common formats.

There are 2,262 missing values (25.37%). Since this variable represents an identifier rather than a numerical measurement, missing values should not automatically be imputed or cause entire records to be removed.

In [25]:
df_clean.loc[
    df_clean["topology_subunit"].isna(),
    ["pdbid", "name", "topology_subunit", "subunit_segments"]
].head(20)

,pdbid,name,topology_subunit,subunit_segments
5,"=""1zhx""","Oxysterol-binding protein homolog 4, conformat...",NaN,0
99,"=""1dfn""",Neutrophil defensin 3,NaN,0
100,"=""1bh4""",Circulin A,NaN,0
101,"=""1myn""",Drosomycin,NaN,0
102,"=""1jch""",Colicin E3,NaN,0
103,"=""1qwd""",Outer membrane lipoprotein Blc,NaN,0
104,"=""1dvp""",Hepatocyte growth factor-regulated tyrosine ki...,NaN,0
107,"=""2rng""",Big defensin,NaN,0
108,"=""1aa7""","Influenza virus matrix protein M1, structure 1",NaN,0
109,"=""2ie6""",Annexin V,NaN,0


In [26]:
df_clean.loc[
    df_clean["topology_subunit"] == "A in",
    ["pdbid", "name", "topology_subunit", "subunit_segments"]
]

,pdbid,name,topology_subunit,subunit_segments
3880,"=""5zsu""","Volume-regulated anion channel LRRC8A, structu...",A in,24
3881,"=""6bfg""",(S)-mandelate dehydrogenase,A in,0
3882,"=""6c9w""","Lactose permease LacY, structure 7",A in,12
4711,"=""6uiv""","Calcium homeostasis modulator CALHM2, open state",A in,44
4712,"=""6uix""","Calcium homeostasis modulator CALHM2, in two m...",A in,88
4725,"=""6uiw""","Calcium homeostasis modulator CALHM2, inhibite...",A in,44
4787,"=""6vak""",Calcium homeostasis modulator CALHM2,A in,44
4788,"=""6vam""",Calcium homeostasis modulator CALHM2,A in,32


### Topology Subunit Inspection

Inspection of the `topology_subunit` column shows that most values represent chain or subunit identifiers, such as `A`, `B`, and `C`. Some less common formats, including `A in`, are also present and are retained because they may contain additional structural or orientation information.

Initial inspection of records with missing `topology_subunit` values shows that these entries also have `subunit_segments = 0`. This suggests that the missing values may be related to structures without assigned transmembrane segments rather than being random missing data.

The relationship between missing `topology_subunit` values and `subunit_segments` will therefore be examined across the complete dataset before deciding how these missing values should be handled.

In [27]:
df_clean.loc[
    df_clean["topology_subunit"].isna(),
    "subunit_segments"
].value_counts().sort_index()

subunit_segments
0     2252
1        3
2        2
7        3
12       1
20       1
Name: count, dtype: int64

### Topology Subunit Missing-Value Assessment

The relationship between missing `topology_subunit` values and the number of membrane-spanning segments was examined across all affected records.

Of the 2,262 records with a missing `topology_subunit`, 2,252 (99.56%) have `subunit_segments = 0`. Only 10 records with a missing topology subunit contain one or more membrane-spanning segments.

This indicates that missing `topology_subunit` values are strongly associated with records without assigned membrane-spanning segments and therefore do not appear to represent random missing data. The missing values will be retained rather than imputed, as assigning a chain or subunit identifier would introduce information that is not present in the original dataset.

In [28]:
df_clean.loc[
    df_clean["topology_subunit"].isna()
    & (df_clean["subunit_segments"] > 0),
    ["pdbid", "name", "topology_subunit", "subunit_segments"]
]

,pdbid,name,topology_subunit,subunit_segments
266,"=""4b19""",Toxic peptide of toxin-antitoxin system (PEPA1),NaN,1
2560,"=""2mts""","P7 protein (747-809), structure 2",NaN,2
2572,"=""4u4g""","Glutamate receptor 2, with partial agonist, st...",NaN,12
2939,"=""5dsg""","Muscarinic acetylcholine receptor M4, inactive...",NaN,7
2940,"=""5cxv""","Muscarinic acetylcholine receptor M1, inactive...",NaN,7
2946,"=""4z0w""",Gichigamin,NaN,2
3494,"=""5osc""","GLIC-GABAAR alpha1 chimera, structure 2",NaN,20
4099,"=""6mf8""","T-cell receptor alpha chain C region, TM segment",NaN,1
4236,"=""6gnz""","Plantaricin S, alpha subunit",NaN,1
4876,"=""6kp6""","Muscarinic acetylcholine receptor M4, inactive...",NaN,7


The 10 exceptional records with missing `topology_subunit` but non-zero `subunit_segments` were also inspected. These represent a small and diverse set of membrane-associated structures, with no clear pattern that would allow the missing subunit identifiers to be reconstructed reliably.

Therefore, no values are imputed and these records are retained in the dataset.

### Topology Subunit Missing-Value Assessment

The `topology_subunit` column contains 2,262 missing values, representing 25.37% of the dataset. Since this variable identifies the chain or subunit associated with the membrane topology, the missing values were investigated before deciding how they should be handled.

Comparison with `subunit_segments` showed that 2,252 of the 2,262 records with missing `topology_subunit` values (99.56%) have `subunit_segments = 0`. This indicates that the missing topology information is strongly associated with structures for which no membrane-spanning segments are recorded, rather than representing random missing data.

The remaining 10 records have between 1 and 20 membrane-spanning segments. These exceptions were inspected individually and represent a small and diverse group of membrane-associated structures. No consistent pattern was identified that would allow the missing subunit identifiers to be reconstructed reliably.

Therefore, the missing `topology_subunit` values will be retained without imputation. Replacing them with the most frequent category or another assumed identifier could introduce information that is not supported by the original dataset. The 10 exceptional records will also be retained, as the absence of a topology subunit identifier alone does not justify removing otherwise valid observations.

### Thickness and Tilt Error Inspection

The `thicknesserror` and `tilterror` variables contain a small number of missing values. Since these variables represent uncertainties associated with membrane thickness and tilt measurements, the missing records are inspected before deciding whether any treatment is required.

In [29]:
df_clean[
    df_clean["thicknesserror"].isna() |
    df_clean["tilterror"].isna()
][
    [
        "pdbid",
        "name",
        "thickness",
        "thicknesserror",
        "tilt",
        "tilterror"
    ]
]

,pdbid,name,thickness,thicknesserror,tilt,tilterror
545,"=""1xq8""",Alpha-synuclein,11.3,NaN,83,4.0
3561,"=""6rd4""",F-ATP synthase,27.6,NaN,0,0.0
4980,"=""6tu2""",Annexin A11,2.3,NaN,88,1.0
6018,"=""7k02""",BAK dimer activated by detergent,10.0,NaN,89,0.0
6132,"=""6b3i""",Annexin A13,1.0,NaN,91,NaN
6134,"=""2d4c""",Endophilin-A1,0.6,NaN,90,NaN
6135,"=""2c08""",Endophilin-A1,1.1,NaN,90,NaN
6136,"=""1uru""",Endophilin-A,1.0,NaN,89,NaN
6137,"=""1zww""",Endophilin-A1,5.5,NaN,88,NaN
6138,"=""2z0v""",Endophilin-A3,3.4,NaN,87,NaN


In [30]:
print("Missing thicknesserror:", df_clean["thicknesserror"].isna().sum())
print("Missing tilterror:", df_clean["tilterror"].isna().sum())

print(
    "Both missing:",
    (
        df_clean["thicknesserror"].isna()
        & df_clean["tilterror"].isna()
    ).sum()
)

Missing thicknesserror: 35
Missing tilterror: 29
Both missing: 29


In [31]:
df_clean.loc[
    df_clean["thicknesserror"].isna(),
    ["id", "pdbid", "name", "thickness", "tilt"]
].tail(40)

,id,pdbid,name,thickness,tilt
545,549,"=""1xq8""",Alpha-synuclein,11.3,83
3561,3638,"=""6rd4""",F-ATP synthase,27.6,0
4980,5111,"=""6tu2""",Annexin A11,2.3,88
6018,6605,"=""7k02""",BAK dimer activated by detergent,10.0,89
6132,6790,"=""6b3i""",Annexin A13,1.0,91
6134,6792,"=""2d4c""",Endophilin-A1,0.6,90
6135,6793,"=""2c08""",Endophilin-A1,1.1,90
6136,6794,"=""1uru""",Endophilin-A,1.0,89
6137,6795,"=""1zww""",Endophilin-A1,5.5,88
6138,6796,"=""2z0v""",Endophilin-A3,3.4,87


### Thickness and Tilt Error Assessment

The `thicknesserror` and `tilterror` variables contain 35 (0.39%) and 29 (0.33%) missing values, respectively. The corresponding `thickness` and `tilt` measurements remain available for these records.

All 29 records with missing `tilterror` also have a missing `thicknesserror`. A further six records have a missing `thicknesserror` while retaining a reported `tilterror`. Inspection of the affected records also shows that many of the missing values occur together in a distinct group of membrane-associated structures rather than being distributed uniformly throughout the dataset.

Because these columns represent uncertainty estimates associated with the reported thickness and tilt values, replacing the missing values with a mean, median, or other estimated value would introduce uncertainty measurements that were not reported in the original dataset.

Therefore, the missing `thicknesserror` and `tilterror` values will be retained as `NaN`. The corresponding records will not be removed because their primary `thickness` and `tilt` measurements remain available for analysis.

### UniProt Code Inspection

The `uniprotcode` column contains 334 missing values (3.75%). Since UniProt codes are biological database identifiers rather than measured variables, the missing entries are inspected before deciding how they should be handled.

In [32]:
df_clean.loc[
    df_clean["uniprotcode"].isna(),
    ["pdbid", "name", "species_name_cache", "uniprotcode"]
].head(20)

,pdbid,name,species_name_cache,uniprotcode
98,"=""1grm""","Gramicidin A, head-to-head dimer, right-handed",Brevibacillus brevis,NaN
166,"=""1ee7""",Chrysospermin C,Hypomyces chrysospermus,NaN
214,"=""1r9u""","Zervamicin IIb, transmembrane orientation",Emericellopsis salmosynnemata,NaN
241,"=""1ozy""","Snake phospholipase A2, group I",Micropechis ikaheka,NaN
527,"=""1smz""",Transportan,Designed proteins,NaN
533,"=""1ih9""",Zervamicin IIb,Emericellopsis salmosynnemata,NaN
534,"=""1m24""",Trichotoxin a50e,Hypocrea rufa,NaN
535,"=""1ob4""",Cephaibol A,Acremonium tubakii,NaN
536,"=""1ob6""",Cephaibol B,Acremonium tubakii,NaN
537,"=""1ob7""",Cephaibol C,Acremonium tubakii,NaN


### UniProt Code Missing-Value Assessment

The `uniprotcode` column contains 334 missing values, representing 3.75% of the dataset. Inspection of affected records shows that they include a diverse range of structures, including naturally occurring peptides, designed proteins, and other membrane-associated molecules.

Since UniProt codes are biological database identifiers rather than measured variables, a missing identifier cannot be reliably estimated or replaced using information from other records. Imputing the most common identifier or another artificial value would therefore introduce incorrect biological information.

The missing `uniprotcode` values will consequently be retained as `NaN`, and the corresponding records will remain in the dataset because the absence of a UniProt identifier does not prevent their other structural and membrane-related variables from being analysed.

### Duplicate Record Inspection

The dataset is checked for duplicate records to determine whether any observations are repeated. Duplicate entries could bias subsequent summary statistics and visualisations if they represent unintended repetitions of the same observation.

In [33]:
df_clean.duplicated().sum()

0

No exact duplicate rows were identified in the dataset. Therefore, no records are removed based on complete-row duplication.

However, individual PDB structures may potentially occur more than once while containing different associated information. The `pdbid` column will therefore be examined separately for repeated identifiers.

In [34]:
df_clean["pdbid"].duplicated().sum()

1

In [35]:
duplicate_pdbids = df_clean[
    df_clean["pdbid"].duplicated(keep=False)
].sort_values("pdbid")

duplicate_pdbids

,id,ordering,family_name_cache,species_name_cache,membrane_name_cache,name,pdbid,resolution,topology_subunit,topology_show_in,...,superfamily_id,classtype_id,type_id,secondary_representations_count,structure_subunits_count,citations_count,created_at,updated_at,uniprotcode,resolution_numeric
5063,5194,6891.0,"Retinol binding protein-like (n=8,S=12)",Homo sapiens,Secreted,"Retinol binding protein 4, with DNA-binding pr...","=""6qba""",1.8,A,True,...,50,3,2,0,0,0,2020-06-20 01:39:50 UTC,2023-02-22 21:56:08 UTC,DN7A_SACS2 RET4_HUMAN,1.8
5208,5339,6892.0,"Retinol binding protein-like (n=8,S=12)",Homo sapiens,Secreted,"Retinol-binding protein 4, with DNA-binding pr...","=""6qba""",1.8,A,True,...,50,3,2,0,0,0,2020-07-25 22:09:19 UTC,2023-02-22 21:56:50 UTC,DN7A_SACS2 RET4_HUMAN,1.8


In [36]:
duplicate_pdbids.loc[:, duplicate_pdbids.nunique(dropna=False) > 1].T

,5063,5208
id,5194,5339
ordering,6891.0,6892.0
name,"Retinol binding protein 4, with DNA-binding pr...","Retinol-binding protein 4, with DNA-binding pr..."
created_at,2020-06-20 01:39:50 UTC,2020-07-25 22:09:19 UTC
updated_at,2023-02-22 21:56:08 UTC,2023-02-22 21:56:50 UTC


In [37]:
df_clean = df_clean.drop_duplicates(
    subset="pdbid",
    keep="first"
)

print("Dataset shape:", df_clean.shape)
print("Duplicated PDB IDs:", df_clean["pdbid"].duplicated().sum())

Dataset shape: (8914, 29)
Duplicated PDB IDs: 0


### Duplicate Record Assessment

No exact duplicate rows were identified in the dataset. However, examination of the `pdbid` column identified one repeated PDB identifier (`6qba`).

The two records were compared across all variables. Differences were limited to database-level metadata (`id`, `ordering`, `created_at`, and `updated_at`) and a minor formatting difference in the protein name. The biological and structural information was otherwise identical.

The second occurrence was therefore treated as a duplicate database entry and removed, while the first occurrence was retained. This reduced the dataset from 8,915 to 8,914 records and prevents the same structure from contributing twice to subsequent analyses.

### PDB Identifier Formatting

Inspection of the `pdbid` column shows that identifiers contain additional formatting characters, for example `="1qjp"` rather than the standard PDB identifier `1qjp`.

These characters are removed to produce consistent identifiers that can be used more easily for analysis and database referencing.

In [38]:
df_clean["pdbid"] = (
    df_clean["pdbid"]
    .str.replace('="', '', regex=False)
    .str.replace('"', '', regex=False)
    .str.strip()
)

In [39]:
df_clean["pdbid"].head(10)

0    1qjp
1    1qj8
2    1p4t
3    3lbw
4    2x55
5    1zhx
6    1k24
7    1uyn
8    1qd6
9    1tly
Name: pdbid, dtype: object

In [40]:
df_clean["pdbid"].str.len().value_counts()

pdbid
4    8914
Name: count, dtype: int64

### PDB Identifier Cleaning Result

The additional formatting characters were successfully removed from the `pdbid` column. All 8,914 remaining PDB identifiers now contain exactly four characters, providing a consistent format for subsequent analysis and external database referencing.

### Categorical Data Consistency

The main categorical variables are checked for formatting inconsistencies such as leading or trailing whitespace. Such inconsistencies can cause biologically identical categories to be treated as separate groups during analysis.

In [41]:
categorical_cols = [
    "family_name_cache",
    "species_name_cache",
    "membrane_name_cache"
]

for col in categorical_cols:
    whitespace_count = (
        df_clean[col].astype(str) !=
        df_clean[col].astype(str).str.strip()
    ).sum()
    
    print(f"{col}: {whitespace_count} values with leading/trailing whitespace")

family_name_cache: 15 values with leading/trailing whitespace
species_name_cache: 18 values with leading/trailing whitespace
membrane_name_cache: 0 values with leading/trailing whitespace


In [42]:
df_clean["family_name_cache"] = df_clean["family_name_cache"].str.strip()
df_clean["species_name_cache"] = df_clean["species_name_cache"].str.strip()

In [43]:
for col in categorical_cols:
    whitespace_count = (
        df_clean[col].astype(str) !=
        df_clean[col].astype(str).str.strip()
    ).sum()
    
    print(f"{col}: {whitespace_count} values with leading/trailing whitespace")

family_name_cache: 0 values with leading/trailing whitespace
species_name_cache: 0 values with leading/trailing whitespace
membrane_name_cache: 0 values with leading/trailing whitespace


The categorical consistency check identified 15 values in `family_name_cache` and 18 values in `species_name_cache` containing leading or trailing whitespace. No such inconsistencies were detected in `membrane_name_cache`.

The affected values were standardized using whitespace stripping. A subsequent validation confirmed that no leading or trailing whitespace remained in the three categorical variables. This prevents identical biological categories from being treated as separate groups during subsequent analysis.

### Membrane Category Distribution

The distribution of the `membrane_name_cache` variable is examined to understand the main membrane environments represented in the dataset and to identify any obvious inconsistencies in category naming.

In [44]:
df_clean["membrane_name_cache"].value_counts()

membrane_name_cache
Eykaryo. plasma         3316
Secreted                1359
Gram-neg. inner         1251
Endoplasm. reticulum     548
Gram-neg. outer          421
Gram-pos. inner          374
Mitochon. inner          347
Thylakoid                192
Undefined                185
Archaebac.               179
Viral                    176
Lysosome                 102
Golgi                    102
Endosome                  97
Vacuole                   82
Mitochon. outer           69
Vesicle                   42
Peroxisome                27
Nuclear outer             16
Gram-pos. outer           10
Chloroplast inner          7
Nuclear inner              6
Chloroplast outer          4
Granule                    2
Name: count, dtype: int64

The `membrane_name_cache` column contains 24 membrane categories. Inspection of their frequencies did not reveal obvious duplicate categories caused by inconsistent capitalization or whitespace. The original category labels, including database-specific abbreviations, were therefore retained.

The dataset is dominated by `Eykaryo. plasma`, followed by `Secreted` and `Gram-neg. inner`. This unequal representation should be considered when comparing membrane categories during subsequent analysis.

### Numerical Variable Validation

Key numerical variables describing membrane-protein properties are examined for potentially invalid or unusual values. Minimum and maximum values are inspected before deciding whether any observations require removal or further investigation.

In [45]:
numerical_cols = [
    "thickness",
    "thicknesserror",
    "subunit_segments",
    "tilt",
    "tilterror",
    "gibbs",
    "resolution_numeric"
]

df_clean[numerical_cols].agg(["min", "max", "mean", "median"]).T

,min,max,mean,median
thickness,0.00,41.8,22.220989,29.65
thicknesserror,-3.00,180.0,1.242876,1.00
subunit_segments,0.00,389.0,14.134283,10.00
tilt,0.00,91.0,23.621943,7.00
tilterror,-4.00,1039.0,3.440293,1.00
gibbs,-691.00,165.5,-89.151604,-71.55
resolution_numeric,0.54,37.0,3.064330,3.00


#### Inspection of Unusual Error Values

The numerical summary identified potentially unusual values in `thicknesserror` and `tilterror`, including negative values and very large maximum values. These observations are inspected individually before determining whether they represent data errors or valid values from the source dataset.

In [46]:
unusual_errors = df_clean[
    (df_clean["thicknesserror"] < 0) |
    (df_clean["thicknesserror"] > 10) |
    (df_clean["tilterror"] < 0) |
    (df_clean["tilterror"] > 20)
][
    [
        "pdbid",
        "name",
        "thickness",
        "thicknesserror",
        "tilt",
        "tilterror"
    ]
]

unusual_errors

,pdbid,name,thickness,thicknesserror,tilt,tilterror
258,1dan,Coagulation factor VIIa,4.2,6.0,25,26.0
259,1nl2,Prothrombin,3.9,3.0,18,24.0
314,3sja,"ATPase Get3, complex with Get1 cytosolic domain",6.4,1.6,89,45.0
356,1cwa,Cyclosporin,10.2,2.5,82,53.0
505,2xu0,Erythrocyte membrane protein 1,3.3,6.0,75,45.0
...,...,...,...,...,...,...
7958,7knd,"C1B domain of protein kinase C, apo form",2.4,1.2,25,22.0
8458,7uw5,"EcMscK, G924S, a closed conformation",29.3,100.0,12,0.0
8656,8hjc,Cysteine-rich peptide bidentatide,6.9,1.8,50,26.0
8657,8hjd,Cysteine-rich peptide bidentatide with glycation,6.8,1.8,52,23.0


The inspection identified 199 records meeting the deliberately broad thresholds used to flag unusual uncertainty values. These thresholds were used for exploratory inspection only and do not define invalid observations.

Several structures have relatively large uncertainty estimates for membrane thickness or tilt. These values may reflect poorly constrained membrane orientations or other characteristics of the original structural calculations. The dataset also contains a small number of negative error values, which are unusual for uncertainty measurements.

Because there is insufficient evidence to determine that these values are data-entry errors, the original values are retained rather than removed, capped, or imputed. The `thicknesserror` and `tilterror` variables will therefore be treated cautiously and will not form the primary basis of the subsequent membrane-protein analysis.

In [47]:
print("Thickness below 0:", (df_clean["thickness"] < 0).sum())
print("Tilt below 0:", (df_clean["tilt"] < 0).sum())
print("Tilt above 90:", (df_clean["tilt"] > 90).sum())
print("Subunit segments below 0:", (df_clean["subunit_segments"] < 0).sum())

Thickness below 0: 0
Tilt below 0: 0
Tilt above 90: 2
Subunit segments below 0: 0


In [48]:
df_clean.loc[
    df_clean["tilt"] > 90,
    ["pdbid", "name", "membrane_name_cache", "thickness", "tilt", "tilterror"]
]

,pdbid,name,membrane_name_cache,thickness,tilt,tilterror
6132,6b3i,Annexin A13,Eykaryo. plasma,1.0,91,NaN
6146,3q0k,PACSIN 2,Endosome,3.6,91,NaN


Two records were identified with tilt values slightly above 90°, both reporting a value of 91°. Inspection showed that these correspond to Annexin A13 and PACSIN 2.

As these values are only marginally above 90° and originate from the source dataset, there is insufficient evidence to classify them as erroneous. The values are therefore retained without modification.

The remaining core numerical variables contained no negative values for membrane thickness, tilt, or number of subunit segments. No additional records were removed during numerical validation.

### Data Cleaning Summary

The dataset was systematically assessed for missing values, duplicate observations, identifier formatting, categorical consistency, and potentially unusual numerical values.

The main cleaning decisions were:

- Columns containing no information were removed.
- Missing values were assessed according to their biological and analytical context rather than automatically imputed.
- Structural resolution was converted into a numerical variable while preserving the original resolution information.
- One duplicated PDB structure was identified, inspected, and removed.
- PDB identifiers were standardized to their four-character format.
- Leading and trailing whitespace was removed from affected family and species categories.
- Unusual numerical and uncertainty values were inspected and retained where there was insufficient evidence that they represented errors.

Following cleaning, the dataset contains 8,914 unique structural records and is ready for exploratory data analysis.

In [49]:
df_clean.to_csv("data/proteins_cleaned.csv", index=False)

In [50]:
df_clean.shape

(8914, 29)